# Baseline multimodal network (audio + text), frozen-first

One notebook, top to bottom: data -> frozen encoders (WavLM audio, DeBERTa text) -> multimodal PyTorch net with **modality dropout** -> speaker-held-out cross-validation with **audio-only / text-only / fused** ablations -> saved checkpoint.

Follows `docs/finetuning_plan.md` section 7 (audio branch with learnable layer weights, text branch, scalar acoustic features, fusion with modality dropout, masked heads). Two honest notes up front:
- **Frozen encoders + cached features** are the right first step with a small labelled set: training the head takes seconds, and results are comparable across runs. Set `UNFREEZE_TOP = N` in the config cell to fine-tune the top N WavLM layers instead (slower, needs a GPU).
- **RAVDESS has only two fixed sentences**, so its text carries no emotion. That is a useful sanity check (text-only must be near chance) but it cannot show what text adds. Add a dataset with real emotional text (MELD, see `docs/research/multimodal_datasets.md`) in the data cell to get a meaningful audio-vs-text comparison.

Run on a laptop (CPU/MPS, about 10-15 min with `QUICK = False`) or Colab (GPU). `QUICK = True` does a 2-minute smoke run on 6 actors.

In [ ]:
# ---- Config ---------------------------------------------------------------------------------
QUICK = True            # True: 6 actors, few epochs (smoke test). False: all 24 actors.
DATASETS = ["ravdess", "meld"]   # "ravdess" (acted, fixed text) and/or "meld" (TV dialogue: audio + real emotional text)
MELD_SPLITS = ["dev"] if QUICK else ["dev", "test"]   # add "train" for the full 9,989 clips (+1.1 GB, ~15 min more feature extraction)
SEED = 0
UNFREEZE_TOP = 0        # 0 = frozen encoders (cached features). >0 not implemented in this cached pipeline.
D_MODEL = 256
EPOCHS = 8 if QUICK else 40
P_MOD_DROP = 0.3        # probability of dropping each modality (never both) while training
N_FOLDS = 3 if QUICK else 6

import os, sys, json, time, random, hashlib, zipfile, urllib.request, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
REPO = Path("..") if (Path("..") / "backend").exists() else None      # running inside the repo?
WORK = (REPO / "data" / "external") if REPO else Path("work")
OUT = (REPO / "results") if REPO else Path("results")
MODELS = (REPO / "data" / "models") if REPO else Path("models")
for p in (WORK, OUT, MODELS, WORK / "cache"):
    p.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| repo:", bool(REPO), "| torch", torch.__version__)

In [ ]:
# ---- Install (Colab / fresh machine). Skipped when everything is already importable. -----------
try:
    import pandas, transformers, librosa, soundfile, sklearn, sentencepiece, google.protobuf  # noqa: F401
except ImportError:
    %pip install -q pandas transformers librosa soundfile scikit-learn sentencepiece protobuf tqdm huggingface_hub

## 1. Data
Every adapter returns a DataFrame with: `path`, `text`, `emotion`, `arousal` (high/low), `speaker`, `group`, `dataset`. Splits are always **by `group`**: the actor for RAVDESS, the TV **episode** for MELD (MELD's six main characters appear in every split, so speaker-disjoint is impossible there; episode-disjoint at least removes shared scenes and dialogue, and the speaker overlap is a stated limitation).

In [ ]:
# ---- RAVDESS (CC BY-NC-SA 4.0, Zenodo 1188976, acted speech, 24 actors) -----------------------
RAV_URL = "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip"
RAV_MD5 = "bc696df654c87fed845eb13823edef8a"   # verified in docs/licences.md workflow
EMO = {1: "neutral", 2: "calm", 3: "happy", 4: "sad", 5: "angry", 6: "fearful", 7: "disgust", 8: "surprised"}
HIGH = {"happy", "angry", "fearful", "surprised"}
STATEMENT = {1: "Kids are talking by the door", 2: "Dogs are sitting by the door"}

def md5(p, n=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(n), b""):
            h.update(b)
    return h.hexdigest()

def load_ravdess():
    root = WORK / "ravdess" / "speech"
    if not any(root.rglob("*.wav")):
        root.mkdir(parents=True, exist_ok=True)
        z = WORK / "ravdess" / "Audio_Speech_Actors_01-24.zip"
        if not z.exists():
            print("downloading RAVDESS (~200 MB) ..."); urllib.request.urlretrieve(RAV_URL + "?download=1", z)
        assert md5(z) == RAV_MD5, "RAVDESS md5 mismatch"
        zipfile.ZipFile(z).extractall(root)
    rows = []
    for p in sorted(root.rglob("*.wav")):
        f = [int(x) for x in p.stem.split("-")]
        e = EMO[f[2]]
        rows.append(dict(path=str(p), text=STATEMENT[f[4]], emotion=e, arousal="high" if e in HIGH else "low",
                         speaker=f"rav_{f[6]:02d}", group=f"rav_{f[6]:02d}", dataset="ravdess"))
    return pd.DataFrame(rows)

MELD_MAP = {"anger": "angry", "disgust": "disgust", "fear": "fearful", "joy": "happy", "neutral": "neutral", "sadness": "sad", "surprise": "surprised"}

def load_meld():
    # MELD (Friends TV dialogue, 7 emotions, text + audio). HF repo ajyy/MELD_audio: CSVs + archive/<split>.tar.gz of
    # <split>/dia{D}_utt{U}.flac. The repo's loading script is bypassed (it has a bytes-key bug); we read the files directly.
    # Licence: GPL-3.0 tag on the card (card field says 'More Information Needed'); TV audio. Check before any product use.
    import tarfile
    from huggingface_hub import hf_hub_download
    rows = []
    for sp in MELD_SPLITS:
        meta = pd.read_csv(hf_hub_download("ajyy/MELD_audio", f"{sp}.csv", repo_type="dataset"))
        dest = WORK / "meld"; (dest / sp).mkdir(parents=True, exist_ok=True)
        if not any((dest / sp).glob("*.flac")):
            tgz = hf_hub_download("ajyy/MELD_audio", f"archive/{sp}.tar.gz", repo_type="dataset")
            with tarfile.open(tgz) as t: t.extractall(dest)
        for _, r in meta.iterrows():
            p = dest / sp / f"dia{r['Dialogue_ID']}_utt{r['Utterance_ID']}.flac"
            e = MELD_MAP.get(str(r["Emotion"]).lower())
            if p.exists() and e:
                rows.append(dict(path=str(p), text=str(r["Utterance"]), emotion=e, arousal="high" if e in HIGH else "low",
                                 speaker=f"meld_{r['Speaker']}", group=f"meld_s{r['Season']}e{r['Episode']}", dataset="meld"))
    return pd.DataFrame(rows)

ADAPTERS = {"ravdess": load_ravdess, "meld": load_meld}
df = pd.concat([ADAPTERS[d]() for d in DATASETS], ignore_index=True)
if QUICK:
    keep = df[df.dataset == "ravdess"].group.drop_duplicates().sort_values().iloc[:6]
    df = df[(df.dataset != "ravdess") | df.group.isin(keep)].reset_index(drop=True)
print(len(df), "clips,", df.group.nunique(), "groups"); print(df.groupby("dataset").emotion.value_counts().unstack(0).fillna(0).astype(int))

## 2. Frozen encoders -> cached features
Audio: WavLM-base-plus, all 13 hidden layers, mean+std pooled over time (the net learns a weighted sum over layers). Text: DeBERTa-v3-base mean-pooled. Scalar acoustic features: duration, loudness, peak, clipping, pause fraction, zero-crossing rate (intensity is information, so loudness is NOT normalised away).

In [ ]:
import librosa
from transformers import AutoModel, AutoTokenizer, WavLMModel
SR = 16_000

def scalar_feats(y):
    fr = librosa.util.frame(y, frame_length=400, hop_length=160)
    rms = np.sqrt((fr ** 2).mean(0) + 1e-12); db = 20 * np.log10(rms + 1e-6)
    active = db > max(np.percentile(db, 10) + 10, np.percentile(db, 95) - 25)
    zcr = float(np.mean(np.abs(np.diff(np.signbit(y).astype(np.int8)))))
    return [len(y) / SR, float(db[active].mean() if active.any() else db.mean()), float(np.abs(y).max()),
            float(np.mean(np.abs(y) >= 0.99)), float(1 - active.mean()), zcr]

def cache_path(name): return WORK / "cache" / f"{name}_{'quick' if QUICK else 'full'}.npz"

def extract_audio(df):
    cp = cache_path("wavlm_" + "_".join(DATASETS))
    if cp.exists():
        z = np.load(cp, allow_pickle=True)
        if list(z["paths"]) == list(df.path):
            return z["A"], z["S"]
    enc = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").eval().to(DEVICE)
    A, S = [], []
    for i, p in enumerate(df.path):
        y, _ = librosa.load(p, sr=SR); y, _ = librosa.effects.trim(y, top_db=30)
        S.append(scalar_feats(y))
        x = (y - y.mean()) / (y.std() + 1e-7)
        with torch.inference_mode():
            hs = enc(torch.from_numpy(x)[None].to(DEVICE), output_hidden_states=True).hidden_states
        h = torch.stack(hs)[:, 0]                                   # (13, T, 768)
        A.append(torch.stack([h.mean(1), h.std(1)], 1).cpu().numpy().astype(np.float16))  # (13, 2, 768)
        if i % 200 == 0: print("audio", i, "/", len(df))
    A, S = np.stack(A), np.array(S, dtype=np.float32)
    np.savez(cp, A=A, S=S, paths=np.array(df.path)); del enc
    return A, S

def extract_text(df):
    tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
    enc = AutoModel.from_pretrained("microsoft/deberta-v3-base").eval().to(DEVICE)
    uniq = sorted(df.text.unique()); out = {}
    for i in range(0, len(uniq), 32):
        b = tok(uniq[i:i + 32], return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
        with torch.inference_mode():
            h = enc(**b).last_hidden_state; m = b["attention_mask"].unsqueeze(-1)
            e = ((h * m).sum(1) / m.sum(1)).cpu().numpy()
        out.update(dict(zip(uniq[i:i + 32], e)))
    return np.stack([out[t] for t in df.text]).astype(np.float32)

t0 = time.time()
A, S = extract_audio(df); T = extract_text(df)
print("features:", A.shape, S.shape, T.shape, f"{time.time() - t0:.0f}s")

## 3. The multimodal network
Audio branch: learnable softmax weights over the 13 WavLM layers, then a projection. Text branch: projection. Scalar branch: small MLP. Fusion: concat of the three embeddings plus the two modality-present flags -> MLP -> heads (emotion, arousal). **Modality dropout** while training makes audio-only and text-only predictions come from the same model; at evaluation a modality is switched off by its mask.

In [ ]:
class MMNet(nn.Module):
    def __init__(self, n_emo, d=D_MODEL, n_layers=13, n_scalar=6):
        super().__init__()
        self.lw = nn.Parameter(torch.zeros(n_layers))
        self.a = nn.Sequential(nn.LayerNorm(2 * 768), nn.Linear(2 * 768, d), nn.GELU(), nn.Dropout(0.2))
        self.t = nn.Sequential(nn.LayerNorm(768), nn.Linear(768, d), nn.GELU(), nn.Dropout(0.2))
        self.s = nn.Sequential(nn.LayerNorm(n_scalar), nn.Linear(n_scalar, 32), nn.GELU())
        self.fuse = nn.Sequential(nn.Linear(2 * d + 32 + 2, d), nn.GELU(), nn.Dropout(0.3))
        self.emo, self.aro = nn.Linear(d, n_emo), nn.Linear(d, 2)

    def forward(self, A, T, S, ma, mt):
        w = torch.softmax(self.lw, 0)[None, :, None]
        za = self.a((A * w[..., None]).sum(1).flatten(1)) * ma[:, None]      # (B, 13, 2, 768) -> (B, 1536)
        zt = self.t(T) * mt[:, None]
        z = self.fuse(torch.cat([za, zt, self.s(S), ma[:, None], mt[:, None]], 1))
        return self.emo(z), self.aro(z)

def modality_masks(B, train, p=P_MOD_DROP, mode="fused"):
    ma = torch.ones(B, device=DEVICE); mt = torch.ones(B, device=DEVICE)
    if mode == "audio": mt = mt * 0
    if mode == "text": ma = ma * 0
    if train:
        r = torch.rand(B, device=DEVICE)
        ma = ma * (r >= p).float()                    # drop audio with prob p
        mt = mt * ((r < p) | (r >= 2 * p)).float()    # drop text with prob p (disjoint from audio drop: never both)
    return ma, mt

## 4. Train + speaker-held-out cross-validation
GroupKFold by `group` (actor / episode; no group in two splits). Each fold trains once; the three evaluation modes (`fused`, `audio`, `text`) use that same model. Checkpoint selection uses 2 validation speakers inside the training folds. Baselines: majority class and logistic regression on the 6 scalar features. Confidence intervals: bootstrap over speakers.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

classes = sorted(df.emotion.unique()); cidx = {c: i for i, c in enumerate(classes)}
y_emo = df.emotion.map(cidx).values; y_aro = (df.arousal == "high").astype(int).values; spk = df.group.values
At, Tt, St = (torch.tensor(x, dtype=torch.float32, device=DEVICE) for x in (A.astype(np.float32), T, S))
yE, yA = torch.tensor(y_emo, device=DEVICE), torch.tensor(y_aro, device=DEVICE)

def predict(model, idx, mode):
    model.eval()
    with torch.no_grad():
        ma, mt = modality_masks(len(idx), False, mode=mode)
        le, la = model(At[idx], Tt[idx], St[idx], ma, mt)
    return le.softmax(-1).cpu().numpy(), la.softmax(-1).cpu().numpy()

def train_fold(tr, va):
    seed_all(SEED); m = MMNet(len(classes)).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2)
    cw = torch.tensor(1.0 / np.bincount(y_emo[tr], minlength=len(classes)).clip(1), dtype=torch.float32, device=DEVICE)
    cw = cw / cw.mean(); best, best_state = -1, None
    for ep in range(EPOCHS):
        m.train(); perm = np.random.permutation(tr)
        for i in range(0, len(perm), 64):
            b = perm[i:i + 64]; ma, mt = modality_masks(len(b), True)
            le, la = m(At[b], Tt[b], St[b], ma, mt)
            loss = F.cross_entropy(le, yE[b], weight=cw, label_smoothing=0.05) + 0.5 * F.cross_entropy(la, yA[b])
            opt.zero_grad(); loss.backward(); opt.step()
        pe, _ = predict(m, va, "fused"); acc = (pe.argmax(1) == y_emo[va]).mean()
        if acc > best: best, best_state = acc, {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state); return m

res = {mode: {"emo": np.zeros(len(df), int), "aro": np.zeros(len(df), int), "p": np.zeros((len(df), len(classes)))} for mode in ("fused", "audio", "text")}
base = {"logreg_scalars": np.zeros(len(df), int)}
for k, (tr, te) in enumerate(GroupKFold(n_splits=N_FOLDS).split(np.zeros(len(df)), y_emo, spk)):
    tr_spk = sorted(set(spk[tr])); va_spk = tr_spk[-2:]
    va = tr[np.isin(spk[tr], va_spk)]; tr2 = tr[~np.isin(spk[tr], va_spk)]
    model = train_fold(tr2, va)
    for mode in res:
        pe, pa = predict(model, te, mode)
        res[mode]["emo"][te] = pe.argmax(1); res[mode]["aro"][te] = pa.argmax(1); res[mode]["p"][te] = pe
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(S[tr], y_emo[tr])
    base["logreg_scalars"][te] = lr.predict(S[te])
    print(f"fold {k + 1}/{N_FOLDS} done, held-out speakers: {sorted(set(spk[te]))[:4]}...")

In [ ]:
rng = np.random.default_rng(SEED)
def boot_ci(correct_by_speaker, n=2000):
    v = np.array(list(correct_by_speaker.values()))
    m = [rng.choice(v, len(v)).mean() for _ in range(n)]
    return [round(float(np.percentile(m, 2.5)), 3), round(float(np.percentile(m, 97.5)), 3)]

def summarise(pred, y):
    per = {s: float((pred[spk == s] == y[spk == s]).mean()) for s in np.unique(spk)}
    return dict(acc=round(float((pred == y).mean()), 3), macro_f1=round(float(f1_score(y, pred, average="macro")), 3), ci95_over_speakers=boot_ci(per))

def ece(p, y, bins=10):
    conf, pred = p.max(1), p.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return round(float(e), 3)

table = {"majority": {"acc": round(float(np.bincount(y_emo).max() / len(y_emo)), 3)},
         "logreg_scalars": summarise(base["logreg_scalars"], y_emo)}
for mode in res:
    table[f"emotion_{mode}"] = summarise(res[mode]["emo"], y_emo)
    table[f"arousal_{mode}"] = summarise(res[mode]["aro"], y_aro)
table["ece_emotion_fused"] = ece(res["fused"]["p"], y_emo)
by_ds = {d: {m: round(float((res[m]["emo"][df.dataset.values == d] == y_emo[df.dataset.values == d]).mean()), 3) for m in res}
         for d in df.dataset.unique()}
print("emotion accuracy per dataset and mode:", by_ds)
pd.DataFrame(table).T

**Reading the table.** `emotion_audio` vs `emotion_fused` vs `emotion_text`: if text-only is near the majority baseline (it must be on RAVDESS, two fixed sentences) and fused is not better than audio-only, text adds nothing on this data. The required question from the plan is *does audio add value for the claim we make*: compare `*_audio` and `*_text` against `*_fused` with the speaker-bootstrap CIs. If the fused network does not beat the simple scalar baseline or the earlier prosody z-scores, ship the simpler one and say so.

In [ ]:
# ---- Final model on all speakers, save checkpoint + results --------------------------------------
all_idx = np.arange(len(df)); spk_all = sorted(set(spk)); va = all_idx[np.isin(spk, spk_all[-2:])]; tr = all_idx[~np.isin(spk, spk_all[-2:])]
final = train_fold(tr, va)
ckpt = MODELS / ("baseline_mm_quick.pt" if QUICK else "baseline_mm.pt")
torch.save({"state_dict": final.state_dict(), "classes": classes, "d_model": D_MODEL, "datasets": DATASETS,
            "encoders": {"audio": "microsoft/wavlm-base-plus (frozen, 13 layers mean+std)", "text": "microsoft/deberta-v3-base (frozen, mean)"},
            "seed": SEED, "quick": QUICK}, ckpt)
out = {"label": "EXPLORATORY", "quick": QUICK, "datasets": DATASETS, "n_clips": len(df), "n_speakers": int(df.speaker.nunique()),
       "folds": N_FOLDS, "emotion_acc_by_dataset": by_ds, "epochs": EPOCHS, "device": DEVICE, "table": table,
       "layer_weights_softmax": [round(float(x), 3) for x in torch.softmax(final.lw.detach().cpu(), 0)]}
(OUT / ("multimodal_cv_quick.json" if QUICK else "multimodal_cv.json")).write_text(json.dumps(out, indent=1))
print("saved", ckpt); print(json.dumps(out["table"], indent=1)[:1500])

## 5. Use it
```python
def predict_clip(wav_path, text):          # same features as training, then the saved network
    ...
```
To plug into the live backend, load the checkpoint in `backend/ml/emotion.py` (it currently loads a logistic probe) and have it extract WavLM stats and DeBERTa pooled text for the utterance (text from the STT result), then fill `emotion_estimate` in the `prosody` details. Keep it off the verdict.

**Next steps for a stronger model:** (1) add a dataset with real emotional text and many speakers (MELD, CREMA-D; see `docs/research/multimodal_datasets.md`) so the text branch and the fusion can be judged; (2) per-speaker history deltas (plan section 7) once a multi-utterance dataset is loaded (AMI sessions); (3) unfreeze the top WavLM layers on a GPU; (4) run the shortcut audit (speaker, loudness, device) and the false-alarm check on calm and anxious speech.